# Chapter 11 - `async` / `await`

#### 1. Mechanical Refresher
An `async def` function returns a coroutine object when called, and its body runs only when the coroutine is awaited by an event loop. `await` pauses the current coroutine while another awaitable finishes, letting the event loop run other ready coroutines during waiting time; this is concurrency, not CPU parallelism.

#### 2. Minimal Working Example

In [ ]:
import asyncio

async def fetch(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return name + " ready"

async def main() -> None:
    first, second = await asyncio.gather(fetch("a", 0.01), fetch("b", 0.01))
    print(first, second)

await main()

Calling `fetch(...)` creates coroutine objects. `asyncio.gather` schedules both, each `await asyncio.sleep` pauses its coroutine, and `main` resumes when both results are ready.

#### 3. Modify Drills

**Modify Drill 1.** Change the names and predict the gathered result order.

In [ ]:
import asyncio

async def tag(name: str) -> str:
    await asyncio.sleep(0.01)
    return name + "!"

actual = await asyncio.gather(tag("x"), tag("y"))
expected = ["x!", "y!"]
assert actual == expected
print("passed:", actual)

**Modify Drill 2.** Await sequentially, then compare the result shape to `gather`.

In [ ]:
import asyncio

async def value(x: int) -> int:
    await asyncio.sleep(0.01)
    return x

first = await value(1)
second = await value(2)
actual = [first, second]
expected = [1, 2]
assert actual == expected
print("passed:", actual)

**Modify Drill 3.** Add an async helper and predict what runs before and after the await.

In [ ]:
import asyncio

async def compute(x: int) -> int:
    await asyncio.sleep(0.01)
    return x * 2

async def wrapper() -> int:
    result = await compute(3)
    return result + 1

assert await wrapper() == 7
print("passed")

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by deleting `await` before `make_value()`. Predict that you get a coroutine object instead of an int, then restore `await`.

In [ ]:
import asyncio

async def make_value() -> int:
    await asyncio.sleep(0.01)
    return 5

actual = await make_value()
assert actual == 5
print("passed:", actual)

**Break-and-Fix Drill 2.** Break it by making `wrapper` a normal `def` while using `await` inside. Predict the syntax error, then keep it as `async def`.

In [ ]:
import asyncio

async def source() -> str:
    await asyncio.sleep(0.01)
    return "ok"

async def wrapper() -> str:
    return await source()

assert await wrapper() == "ok"
print("passed")

#### 5. Self-Verification
Async drills are correct when awaited results match the asserts. Also verify execution order mentally: coroutine objects are created first, awaited operations pause, then the event loop resumes coroutines when their awaited work is ready.

#### 6. Standalone Exercises

**Exercise 1.** Complete an async function that returns a doubled value. Expected behavior: `8`.

In [ ]:
import asyncio

async def async_double(x: int) -> int:
    await asyncio.sleep(0.01)
    return 0  # TODO

actual = await async_double(4)
assert actual == 8
print("passed:", actual)

**Exercise 2.** Gather three async calls. Expected behavior: `[1, 2, 3]`.

In [ ]:
import asyncio

async def identity(x: int) -> int:
    await asyncio.sleep(0.01)
    return x

actual = []  # TODO: await asyncio.gather for 1, 2, and 3.
expected = [1, 2, 3]
assert actual == expected
print("passed:", actual)

**Exercise 3.** Await inside a loop and collect results. Expected behavior: `[2, 4]`.

In [ ]:
import asyncio

async def double_later(x: int) -> int:
    await asyncio.sleep(0.01)
    return x * 2

actual = []
for value in [1, 2]:
    # TODO: append awaited result.
    pass
assert actual == [2, 4]
print("passed:", actual)

**Exercise 4.** Write an async wrapper around an async function. Expected behavior: `'ok!'`.

In [ ]:
import asyncio

async def base() -> str:
    await asyncio.sleep(0.01)
    return "ok"

async def excited() -> str:
    return ""  # TODO: await base and add !

assert await excited() == "ok!"
print("passed")

**Exercise 5.** Preserve result order with `gather`. Expected behavior: `['slow', 'fast']`.

In [ ]:
import asyncio

async def label(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return name

actual = await asyncio.gather(label("slow", 0.02), label("fast", 0.01))
expected = ["slow", "fast"]
assert actual == expected
print("passed:", actual)

#### 7. Applied AI/ML Drill
**ML to Python mirror:** concurrently fetching several synthetic batches is just gathering multiple coroutines that each return one batch. **Python to ML mirror:** async data services, remote feature fetchers, and evaluation workers use this same pattern when waiting on I/O-bound batch retrieval while keeping the event loop free.

**Applied Drill.** Complete concurrent batch fetching. Expected behavior: two batches in request order.

In [ ]:
import asyncio

async def fetch_batch(batch_id: int) -> tuple[int, list[int]]:
    await asyncio.sleep(0.01)
    return batch_id, [batch_id * 10, batch_id * 10 + 1]

batches = []  # TODO: await gather for batch ids 1 and 2.
expected = [(1, [10, 11]), (2, [20, 21])]
assert batches == expected
print("passed:", batches)

#### 8. Common Bugs
- Calling an async function without `await`: the symptom is a coroutine object instead of the result.
- Putting `await` inside a normal `def`: the symptom is a syntax error.
- Expecting CPU speedups from async: async helps with waiting, not CPU-bound work.
- Forgetting that `gather` preserves input order: completion order may differ, but the returned list follows the order of awaitables passed in.

#### 9. Compounding Drill

Combine async with decorators: write a decorator for async functions that awaits the original function and returns its result with a log prefix.

In [ ]:
import asyncio
from functools import wraps

def async_tag(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        result = await func(*args, **kwargs)
        return result  # TODO: prefix with logged:
    return wrapper

@async_tag
async def load_name() -> str:
    await asyncio.sleep(0.01)
    return "batch"

assert await load_name() == "logged:batch"
print("passed")

#### 10. Chapter Project
No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-3
import asyncio

async def async_double(x: int) -> int:
    await asyncio.sleep(0.01)
    return x * 2

assert await async_double(4) == 8

async def identity(x: int) -> int:
    await asyncio.sleep(0.01)
    return x

assert await asyncio.gather(identity(1), identity(2), identity(3)) == [1, 2, 3]

async def double_later(x: int) -> int:
    await asyncio.sleep(0.01)
    return x * 2

actual = []
for value in [1, 2]:
    actual.append(await double_later(value))
assert actual == [2, 4]
print("solutions passed")

In [ ]:
# Standalone Exercises 4-5, Applied Drill, and Compounding Drill
import asyncio
from functools import wraps

async def base() -> str:
    await asyncio.sleep(0.01)
    return "ok"

async def excited() -> str:
    return await base() + "!"

assert await excited() == "ok!"

async def fetch_batch(batch_id: int) -> tuple[int, list[int]]:
    await asyncio.sleep(0.01)
    return batch_id, [batch_id * 10, batch_id * 10 + 1]

batches = await asyncio.gather(fetch_batch(1), fetch_batch(2))
assert batches == [(1, [10, 11]), (2, [20, 21])]

def async_tag(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        result = await func(*args, **kwargs)
        return "logged:" + result
    return wrapper

@async_tag
async def load_name() -> str:
    await asyncio.sleep(0.01)
    return "batch"

assert await load_name() == "logged:batch"
print("solutions passed")